In [2]:
import pandas as pd

df = pd.read_csv("../raw/dft_traffic_counts_raw_counts.csv", low_memory=False)

print(df.head())
print(df.columns.tolist())
print(df.shape)

   count_point_id direction_of_travel  year  count_date  hour  region_id  \
0              51                   S  2004  2004-05-21    11          1   
1              51                   S  2004  2004-05-21    15          1   
2              51                   S  2004  2004-05-21    13          1   
3              51                   N  2004  2004-05-21     7          1   
4              51                   N  2004  2004-05-21    10          1   

  region_name region_ons_code  local_authority_id local_authority_name  ...  \
0  South West       E12000009                   1      Isles of Scilly  ...   
1  South West       E12000009                   1      Isles of Scilly  ...   
2  South West       E12000009                   1      Isles of Scilly  ...   
3  South West       E12000009                   1      Isles of Scilly  ...   
4  South West       E12000009                   1      Isles of Scilly  ...   

  buses_and_coaches LGVs HGVs_2_rigid_axle HGVs_3_rigid_axle  \
0   

In [3]:
keep_cols = [
    "count_point_id",
    "direction_of_travel",
    "count_date",
    "hour",
    "region_name",
    "local_authority_name",
    "road_name",
    "road_category",
    "road_type",
    "latitude",
    "longitude",
    "link_length_km",
    "pedal_cycles",
    "cars_and_taxis",
    "buses_and_coaches",
    "LGVs",
    "all_HGVs",
    "all_motor_vehicles"
]

df = df[keep_cols].copy()

print(df.head())
print(df.shape)

   count_point_id direction_of_travel  count_date  hour region_name  \
0              51                   S  2004-05-21    11  South West   
1              51                   S  2004-05-21    15  South West   
2              51                   S  2004-05-21    13  South West   
3              51                   N  2004-05-21     7  South West   
4              51                   N  2004-05-21    10  South West   

  local_authority_name road_name road_category road_type   latitude  \
0      Isles of Scilly     A3111            PA     Major  49.915015   
1      Isles of Scilly     A3111            PA     Major  49.915015   
2      Isles of Scilly     A3111            PA     Major  49.915015   
3      Isles of Scilly     A3111            PA     Major  49.915015   
4      Isles of Scilly     A3111            PA     Major  49.915015   

   longitude  link_length_km  pedal_cycles  cars_and_taxis  buses_and_coaches  \
0  -6.317138             0.3            12            27.0       

In [4]:
df["count_date"] = pd.to_datetime(df["count_date"])
df["timestamp"] = df["count_date"] + pd.to_timedelta(df["hour"], unit="h")

print(df[["count_date", "hour", "timestamp"]].head())

  count_date  hour           timestamp
0 2004-05-21    11 2004-05-21 11:00:00
1 2004-05-21    15 2004-05-21 15:00:00
2 2004-05-21    13 2004-05-21 13:00:00
3 2004-05-21     7 2004-05-21 07:00:00
4 2004-05-21    10 2004-05-21 10:00:00


In [5]:
london_df = df[df["region_name"].str.lower() == "london"].copy()

print("Original shape:", df.shape)
print("London shape:", london_df.shape)
print(london_df["region_name"].value_counts())
print(london_df["local_authority_name"].nunique())

Original shape: (5113740, 19)
London shape: (414252, 19)
region_name
London    414252
Name: count, dtype: int64
33


In [6]:
agg_df = (
    london_df.groupby(
        ["count_point_id", "timestamp", "local_authority_name", "road_name", "road_category", "road_type"],
        as_index=False
    )[["all_motor_vehicles"]]
    .sum()
)

print(agg_df.head())
print(agg_df.shape)

agg_df.to_csv("../outputs/london_cleaned_counts.csv", index=False)
print("Saved aggregated London dataset.")

   count_point_id           timestamp local_authority_name road_name  \
0            6000 2000-03-27 07:00:00               Barnet        M1   
1            6000 2000-03-27 08:00:00               Barnet        M1   
2            6000 2000-03-27 09:00:00               Barnet        M1   
3            6000 2000-03-27 10:00:00               Barnet        M1   
4            6000 2000-03-27 11:00:00               Barnet        M1   

  road_category road_type  all_motor_vehicles  
0            TM     Major              3580.0  
1            TM     Major              3102.0  
2            TM     Major              3215.0  
3            TM     Major              2486.0  
4            TM     Major              2299.0  
(213612, 7)
Saved aggregated London dataset.


In [7]:
london_df = london_df.dropna(subset=[
    "count_point_id",
    "timestamp",
    "all_motor_vehicles"
])

print(london_df.shape)
print(london_df.isnull().sum())

(414251, 19)
count_point_id               0
direction_of_travel          0
count_date                   0
hour                         0
region_name                  0
local_authority_name         0
road_name                    0
road_category                0
road_type                    0
latitude                     0
longitude                    0
link_length_km          182556
pedal_cycles                 0
cars_and_taxis               0
buses_and_coaches            0
LGVs                         0
all_HGVs                     0
all_motor_vehicles           0
timestamp                    0
dtype: int64


In [8]:
london_df = london_df.sort_values(["count_point_id", "timestamp"]).reset_index(drop=True)

print(london_df[["count_point_id", "timestamp", "all_motor_vehicles"]].head(10))

   count_point_id           timestamp  all_motor_vehicles
0            6000 2000-03-27 07:00:00              2311.0
1            6000 2000-03-27 07:00:00              1269.0
2            6000 2000-03-27 08:00:00              1633.0
3            6000 2000-03-27 08:00:00              1469.0
4            6000 2000-03-27 09:00:00              2270.0
5            6000 2000-03-27 09:00:00               945.0
6            6000 2000-03-27 10:00:00               853.0
7            6000 2000-03-27 10:00:00              1633.0
8            6000 2000-03-27 11:00:00              1305.0
9            6000 2000-03-27 11:00:00               994.0


In [9]:
london_df.to_csv("../outputs/london_cleaned_counts.csv", index=False)
print("Saved cleaned London dataset.")

Saved cleaned London dataset.
